# Training on GENERATED data

# 1. Load dataloaders 

In [ ]:
from data.dataloader import create_dataloaders

train_dataloader, val_dataloader = create_dataloaders(
    csv_path= "../../project_datasets/audio/tubular/parkinsons_generated.csv",
    batch_size= 256,
)

# 2. Load models

- **DenseNet169** – ~14M parameters  
- **EfficientNet-B1** – ~7.8M parameters  
- **MobileNetV3-Large** – ~5.4M parameters  
- **ResNet18** – ~11.7M parameters

In [ ]:
from Models import (
    densenet169,
    efficientnetB1,
    mobilenetV3,
    resnet18
)

models = [
    densenet169.DenseNet1691D(),
    efficientnetB1.EfficientNet1D(),
    mobilenetV3.MobileNet1D_V3(),
    resnet18.ResNet1D()
]

model_names = [
    "DenseNet169",
    "EfficientNetB1",
    "MobileNetV3-Large",
    "ResNet18",
]

# 3. Train models

In [ ]:
from training.trainer import train

for model, model_name in zip(models, model_names):
    train(
        model= model,
        train_dataloader= train_dataloader,
        val_dataloader= val_dataloader,
        
        model_name= model_name,
        run_name= model_name,
        
        epochs= 20,
        max_lr = 1e-3,
    )

In [ ]:
from twilio.rest import Client
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Example: access a variable
account_sid = os.getenv("SID")
auth_token = os.getenv("AUTH_CODE")
from_number = os.getenv("FROM_NUMBER")
to_number = os.getenv("TO_NUMBER")
content = os.getenv("CONTENT")

client = Client(account_sid, auth_token)

message = client.messages.create(
    from_=from_number,
    content_sid=content,
    content_variables='{"1":"12/1","2":"3pm"}',
    to=to_number
)

In [ ]:
# !tensorboard --logdir=runs

# 4. Plot confusion matrix (of the best model)

In [ ]:
import torch
from Models.efficientnetB1 import EfficientNet1D

model = EfficientNet1D()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
load_pretrained = "checkpoints/best.pth"

# load checkpoint
checkpoint = torch.load(load_pretrained, map_location=device)
# load model
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded pretrained model:")
print(f"- val_loss={checkpoint['val_loss']:.4f}")
print(f"- val_acc={checkpoint['val_acc']:.4f}")
print(f"- val_recall={checkpoint['val_recall']:.4f}")
print(f"- val_precision={checkpoint['val_precision']:.4f}")
print(f"- val_f1={checkpoint['val_f1']:.4f}")

In [ ]:
from training.confusion_mat import plot_confusion_matrix

plot_confusion_matrix(
    model=model,
    dataloader=val_dataloader,
    device=device,
    class_names=["Healthy", "PD"],
    # threshold=0.4,
)